Merge Exportov z WOS pre jednotlivé Query

In [ ]:
import pandas as pd
import glob
import os

# =========================
# CHANGE THESE FOR EACH QUERY
# =========================

QUERY_ID = "Q4"
QUERY_LABEL = "market_volatility_GarchHAR_AI_ML_DL_covid"

INPUT_FOLDER = "/content/wos_exports"
OUTPUT_FILE = f"/content/{QUERY_ID}_{QUERY_LABEL}.csv"

# =========================
# COLUMNS TO KEEP
# =========================

wanted_columns = [
    "Authors",
    "Author Full Names",
    "Article Title",
    "Source Title",
    "Publication Year",
    "DOI",
    "Abstract",
    "Author Keywords",
    "Keywords Plus",
    "Document Type",
    "WoS Categories",
    "Research Areas",
    "Times Cited Count",
    "ISSN",
    "Language",
    "IDS Number"
]

# =========================
# LOAD AND MERGE
# =========================

files = glob.glob(os.path.join(INPUT_FOLDER, "*.xls"))

if not files:
    raise FileNotFoundError("No .xls files found in /content/wos_exports")

all_data = []

for file in files:
    filename = os.path.basename(file)
    print("Loading:", filename)

    df = pd.read_excel(file)

    # Clean column names
    df.columns = df.columns.astype(str).str.strip()

    # Keep only useful columns that actually exist in the file
    existing_columns = [col for col in wanted_columns if col in df.columns]
    df = df[existing_columns]

    # Add query information
    df["query_id"] = QUERY_ID
    df["query_label"] = QUERY_LABEL

    all_data.append(df)

merged_df = pd.concat(all_data, ignore_index=True)

# Remove columns that are still completely empty
merged_df = merged_df.dropna(axis=1, how="all")

# Save
merged_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print("Done.")
print("Files loaded:", len(files))
print("Total records:", len(merged_df))
print("Columns kept:", len(merged_df.columns))
print("Saved to:", OUTPUT_FILE)

print("\nFinal columns:")
print(merged_df.columns.tolist())

Loading: savedrecs (4).xls
Done.
Files loaded: 1
Total records: 16
Columns kept: 17
Saved to: /content/Q4_market_volatility_GarchHAR_AI_ML_DL_covid.csv

Final columns:
['Authors', 'Author Full Names', 'Article Title', 'Source Title', 'Publication Year', 'DOI', 'Abstract', 'Author Keywords', 'Keywords Plus', 'Document Type', 'WoS Categories', 'Research Areas', 'ISSN', 'Language', 'IDS Number', 'query_id', 'query_label']


Kontrola počtu po mergnutí.

In [ ]:
df_check = pd.read_csv(OUTPUT_FILE)
len(df_check)

16

Merge všetkých queries a odstránenie duplicít

In [ ]:
import re

# =========================
# SETTINGS
# =========================

INPUT_FOLDER = "/content/wos_queries"

OUTPUT_ALL = "/content/wos_all_records_with_duplicates.csv"
OUTPUT_UNIQUE = "/content/wos_merged_deduplicated.csv"
OUTPUT_DUPLICATES = "/content/wos_duplicates_removed.csv"

query_labels = {
    "Q1": "market_volatility_covid_modeling",
    "Q2": "market_volatility_GarchHar_covid",
    "Q3": "market_volatility_AI_ML_DL_covid",
    "Q4": "market_volatility_GarchHAR_AI_ML_DL_covid"
}

# =========================
# HELPER FUNCTIONS
# =========================

def extract_query_id(filename):
    match = re.search(r"(Q\d+)", filename.upper())
    return match.group(1) if match else "UNKNOWN"

def find_column(df, possible_names):
    col_map = {col.lower().strip(): col for col in df.columns}
    for name in possible_names:
        key = name.lower().strip()
        if key in col_map:
            return col_map[key]
    return None

def clean_doi(x):
    if pd.isna(x):
        return ""
    x = str(x).lower().strip()
    x = x.replace("https://doi.org/", "")
    x = x.replace("http://dx.doi.org/", "")
    x = x.replace("doi:", "")
    if x in ["", "nan", "none", "na", "n/a"]:
        return ""
    return x

def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).lower()
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def clean_year(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    x = x.replace(".0", "")
    return x

def read_csv_safely(file):
    try:
        return pd.read_csv(file, encoding="utf-8-sig")
    except UnicodeDecodeError:
        return pd.read_csv(file, encoding="latin1")

# =========================
# LOAD ALL CSV FILES
# =========================

files = glob.glob(os.path.join(INPUT_FOLDER, "*.csv"))

if not files:
    raise FileNotFoundError(f"No CSV files found in {INPUT_FOLDER}")

dfs = []

for file in files:
    filename = os.path.basename(file)
    print("Loading:", filename)

    df = read_csv_safely(file)
    df.columns = df.columns.astype(str).str.strip()

    query_id = extract_query_id(filename)
    query_label = query_labels.get(query_id, "unknown_query")

    # Add only if missing
    if "query_id" not in df.columns:
        df["query_id"] = query_id

    if "query_label" not in df.columns:
        df["query_label"] = query_label

    df["source_file"] = filename

    dfs.append(df)

all_records = pd.concat(dfs, ignore_index=True)

print("\nTotal files loaded:", len(files))
print("Total records before deduplication:", len(all_records))

# Save raw merged version
all_records.to_csv(OUTPUT_ALL, index=False, encoding="utf-8-sig")

# =========================
# DETECT DEDUPLICATION COLUMNS
# =========================

doi_col = find_column(all_records, ["DOI", "Digital Object Identifier"])

title_col = find_column(all_records, [
    "Article Title",
    "Title",
    "Document Title"
])

year_col = find_column(all_records, [
    "Publication Year",
    "Year Published",
    "PY",
    "Year"
])

authors_col = find_column(all_records, [
    "Authors",
    "Author(s)",
    "AU"
])

wos_id_col = find_column(all_records, [
    "UT",
    "UT (Unique WOS ID)",
    "Unique WOS ID",
    "WoS Accession Number",
    "Accession Number"
])

print("\nDetected columns:")
print("DOI:", doi_col)
print("Title:", title_col)
print("Year:", year_col)
print("Authors:", authors_col)
print("WOS ID:", wos_id_col)

# =========================
# CREATE DEDUPLICATION KEY
# =========================

all_records["doi_clean"] = all_records[doi_col].apply(clean_doi) if doi_col else ""
all_records["title_clean"] = all_records[title_col].apply(clean_text) if title_col else ""
all_records["year_clean"] = all_records[year_col].apply(clean_year) if year_col else ""
all_records["authors_clean"] = all_records[authors_col].apply(clean_text) if authors_col else ""
all_records["wos_id_clean"] = all_records[wos_id_col].apply(clean_text) if wos_id_col else ""

all_records["dedup_key"] = ""

# Priority 1: DOI
mask_doi = all_records["doi_clean"] != ""
all_records.loc[mask_doi, "dedup_key"] = "doi_" + all_records.loc[mask_doi, "doi_clean"]

# Priority 2: WOS unique ID, if available
mask_wos = (all_records["dedup_key"] == "") & (all_records["wos_id_clean"] != "")
all_records.loc[mask_wos, "dedup_key"] = "wos_" + all_records.loc[mask_wos, "wos_id_clean"]

# Priority 3: title + year + authors
mask_title = all_records["dedup_key"] == ""
all_records.loc[mask_title, "dedup_key"] = (
    "title_year_author_"
    + all_records.loc[mask_title, "title_clean"]
    + "_"
    + all_records.loc[mask_title, "year_clean"]
    + "_"
    + all_records.loc[mask_title, "authors_clean"]
)

# Prevent completely empty records from collapsing into one duplicate
empty_key_mask = all_records["dedup_key"].isin([
    "",
    "title_year_author___",
    "title_year_author__"
])

all_records.loc[empty_key_mask, "dedup_key"] = (
    "row_" + all_records.loc[empty_key_mask].index.astype(str)
)

# =========================
# AGGREGATE QUERY INFORMATION
# =========================

query_hits = (
    all_records
    .groupby("dedup_key")["query_id"]
    .apply(lambda x: "; ".join(sorted(set(x.astype(str)))))
    .reset_index()
    .rename(columns={"query_id": "matched_queries"})
)

label_hits = (
    all_records
    .groupby("dedup_key")["query_label"]
    .apply(lambda x: "; ".join(sorted(set(x.astype(str)))))
    .reset_index()
    .rename(columns={"query_label": "matched_query_labels"})
)

file_hits = (
    all_records
    .groupby("dedup_key")["source_file"]
    .apply(lambda x: "; ".join(sorted(set(x.astype(str)))))
    .reset_index()
    .rename(columns={"source_file": "matched_files"})
)

duplicate_counts = (
    all_records
    .groupby("dedup_key")
    .size()
    .reset_index(name="duplicate_count")
)

# =========================
# REMOVE DUPLICATES
# =========================

unique_records = all_records.drop_duplicates(subset="dedup_key", keep="first").copy()

unique_records = unique_records.merge(query_hits, on="dedup_key", how="left")
unique_records = unique_records.merge(label_hits, on="dedup_key", how="left")
unique_records = unique_records.merge(file_hits, on="dedup_key", how="left")
unique_records = unique_records.merge(duplicate_counts, on="dedup_key", how="left")

duplicates_removed = all_records[all_records.duplicated(subset="dedup_key", keep="first")].copy()

# =========================
# SAVE RESULTS
# =========================

unique_records.to_csv(OUTPUT_UNIQUE, index=False, encoding="utf-8-sig")
duplicates_removed.to_csv(OUTPUT_DUPLICATES, index=False, encoding="utf-8-sig")

# =========================
# SUMMARY
# =========================

raw_count = len(all_records)
unique_count = len(unique_records)
duplicates_count = raw_count - unique_count

print("\n=========================")
print("MERGE SUMMARY")
print("=========================")
print("Total records loaded:", raw_count)
print("Unique records:", unique_count)
print("Duplicates removed:", duplicates_count)

print("\nRecords by query:")
print(all_records["query_id"].value_counts().sort_index())

print("\nTop matched query combinations:")
print(unique_records["matched_queries"].value_counts().head(20))

print("\nSaved files:")
print(OUTPUT_ALL)
print(OUTPUT_UNIQUE)
print(OUTPUT_DUPLICATES)

Loading: Q4_market_volatility_GarchHAR_AI_ML_DL_covid.csv
Loading: Q1_market_volatility_covid_modeling.csv
Loading: Q3_market_volatility_AI_ML_DL_covid.csv
Loading: Q2_market_volatility_GarchHar_covid.csv

Total files loaded: 4
Total records before deduplication: 2475

Detected columns:
DOI: DOI
Title: Article Title
Year: Publication Year
Authors: Authors
WOS ID: None

MERGE SUMMARY
Total records loaded: 2475
Unique records: 1864
Duplicates removed: 611

Records by query:
query_id
Q1    1829
Q2     508
Q3     122
Q4      16
Name: count, dtype: int64

Top matched query combinations:
matched_queries
Q1                1251
Q1; Q2             464
Q1; Q3              96
Q2                  28
Q1; Q2; Q3; Q4      16
Q3                   9
Name: count, dtype: int64

Saved files:
/content/wos_all_records_with_duplicates.csv
/content/wos_merged_deduplicated.csv
/content/wos_duplicates_removed.csv
